# DAS Curved Pass Study — CORRECTED

## What was wrong with visualization_2 and visualization_3

**visualization_2 (`compute_xc`)**: it built `phi_grid` with angles sampled along each curve's arc, but then always read the result back at `phi_idx = argmin(|phi_vals - phi_target|)`, where `phi_target` is the straight passer→receiver angle. Since `phi_target` is always itself a member of `phi_vals` (distance 0), that argmin always re-selects the straight-line row — every bulge value silently fell back to the exact same straight-line xC. That is why your plot showed identical 0.081 for every curve.

**visualization_3 (`compute_xc_apex`)**: fixed the "always straight" bug, but replaced it with an **"Apex Risk Proxy"** — it evaluates `defense_cum_prob` at a single point (the arc's midpoint) using the straight-line ray from the ball's start through that one point. It's explicitly a heuristic (the notebook itself says so), not the model's real physics: every other point along the curve is ignored, and the *ball's travel time to that point* is computed as if along a straight ray, not the actual curved path.

**Both notebooks** also computed `xC = attack_prob * (1 - defense_prob)`. The library's own public API (`get_expected_pass_completion`) does **not** do this — it uses `attack_cum_prob` directly. Both quantities already come out of the same competing-risks (survival) model and sum to ≤ 1, so multiplying them double-penalizes.

## The fix

`compute_xc_curved` re-implements the model's actual competing-risks integral (Sec. 2–3 of `accessible_space/core.py`) but walks it along the REAL curved ball path — one point per arc sample — instead of a single straight ray or a single apex point:

1. **Ball time** at each arc sample = cumulative *arc length* / v0 (not straight-line distance), plus the library's own `pass_start_location_offset` / `time_offset_ball` constants.
2. **Player time-to-arrive** at that same (x, y), via the library's own `approx_two_point_time_to_arrive` motion model.
3. **Interception rate** at each point = the library's own fitted logistic — using its `_approximate_sigmoid` (the default `use_efficient_sigmoid=True` path). The *exact* logistic sigmoid gives materially different, thinner-tailed, wrong results — this was a second bug hiding in my first attempt at a fix, caught only by validating against the library.
4. **Integrate** the rate along the curve (trapezoidal rule, in time) to get the true cumulative interception probability at the reception point.

Validated directly against `accessible_space.core.simulate_passes` below (straight-line case + several random scenarios), matching within ~1%.

In [1]:
"""
curved_pass_xc.py
==================
A physically-consistent Expected Completion (xC) calculator for CURVED passes,
built on top of the `accessible-space` DAS engine's own fitted model
(https://github.com/jonas-bischofberger/accessible-space).

WHY THIS FILE EXISTS
---------------------
`accessible_space.core.simulate_passes` only simulates STRAIGHT-line ball
trajectories (a single angle `phi` + speed `v0` per pass). A curved pass isn't
a straight line, so it cannot be represented by one (phi, v0) pair, and the
library has no built-in support for it.

Both `visualization_2_defenders.ipynb` and `visualization_3_corrected.ipynb`
tried to work around this by still calling the straight-line engine and then
picking a single value out of its output:

  - v2 (`compute_xc`): builds `phi_grid` with the curve's mid-arc angles, but
    then always reads back the result at `phi_idx = argmin(|phi_vals - phi_target|)`,
    where `phi_target` is the STRAIGHT chord angle (passer -> receiver). Since
    `phi_target` is always included in `phi_vals` itself, that argmin always
    finds distance 0 and just re-selects the straight-line row - the curve's
    angles are computed and then thrown away. That's why every bulge value
    produced the identical xC = 0.081 in your plot: the curve never actually
    entered the calculation.

  - v3 (`compute_xc_apex`): fixes the "always straight" bug, but replaces it
    with a different approximation - it evaluates `defense_cum_prob` at a
    SINGLE point (the arc's midpoint), using the straight-line ray from the
    ball's start position through that one point. That's a heuristic ("Apex
    Risk Proxy", as the notebook itself calls it), not the model's actual
    physics: it ignores every other point along the curve, and it evaluates
    the defender's own straight-line reach at a made-up "distance", not the
    time it would actually take the ball to get there along the curve.

  - Both also compute `xC = attack_prob * (1 - defense_prob)`. The library's
    own public API (`get_expected_pass_completion`) does NOT do this - it
    uses `attack_cum_prob` directly. `attack_cum_prob` and `defense_cum_prob`
    are already two outcomes of the same competing-risks (survival) model
    that sum to <= 1, so multiplying them together double-counts the risk.

WHAT THIS FILE DOES INSTEAD
-----------------------------
It re-implements the model's actual competing-risks integral (Sec. 2-3 of
`accessible_space/core.py`) but walks it along the REAL curved ball path
(one point per arc sample) instead of a single straight ray:

  1. Ball time-to-reach each arc sample = cumulative arc length / v0
     (+ the model's fitted `time_offset_ball`), i.e. the ball is still
     assumed to travel at constant speed, but along the curve, not a line.
  2. Each player's time-to-arrive at that same (x, y) point, using the
     library's own `approx_two_point_time_to_arrive` motion model
     (accounts for the defender's current velocity/inertia).
  3. Interception RATE at each point = sigmoid(b0 + b1 * (t_player - t_ball)),
     using the model's own fitted b0/b1/factor/factor2 constants.
  4. Integrate that rate along the curve (trapezoidal rule, in TIME) to get
     the cumulative probability that each player has intercepted the ball by
     each point along its curved flight - exactly the `cum_p0` / `pr_cum_prob`
     machinery in `core.py`, just evaluated on an arbitrary path instead of a
     straight ray.

Validation: for a dead-straight "curve" (bulge = 0), this must reproduce the
library's own `simulate_passes(...).attack_cum_prob` almost exactly. See
`validate_against_library()` below - run it before trusting any curved output.
"""

import numpy as np

from accessible_space.core import (
    _DEFAULT_B0, _DEFAULT_B1, _DEFAULT_FACTOR, _DEFAULT_FACTOR2,
    _DEFAULT_TIME_OFFSET_BALL, _DEFAULT_PLAYER_VELOCITY,
    _DEFAULT_KEEP_INERTIAL_VELOCITY, _DEFAULT_USE_MAX, _DEFAULT_V_MAX,
    _DEFAULT_A_MAX, _DEFAULT_INERTIAL_SECONDS, _DEFAULT_TOL_DISTANCE,
    _DEFAULT_PASS_START_LOCATION_OFFSET, _DEFAULT_USE_EFFICIENT_SIGMOID,
    _sigmoid, _approximate_sigmoid, simulate_passes,
)
from accessible_space.motion_models import approx_two_point_time_to_arrive


# ----------------------------------------------------------------------
# Geometry helpers (unchanged from the original notebooks)
# ----------------------------------------------------------------------

def bezier_arc(start, end, bulge, n_points=300):
    """Quadratic Bezier arc from `start` to `end`, bulged sideways by `bulge` * chord length."""
    P0 = np.array(start, dtype=float)
    P2 = np.array(end, dtype=float)
    mid = 0.5 * (P0 + P2)
    chord_vec = P2 - P0
    chord_len = np.linalg.norm(chord_vec)
    perp = np.array([-chord_vec[1], chord_vec[0]]) / chord_len
    P1 = mid + bulge * chord_len * perp
    t = np.linspace(0, 1, n_points)
    x = (1 - t) ** 2 * P0[0] + 2 * t * (1 - t) * P1[0] + t ** 2 * P2[0]
    y = (1 - t) ** 2 * P0[1] + 2 * t * (1 - t) * P1[1] + t ** 2 * P2[1]
    return x, y, P1


def calculate_curvature_2d(x, y):
    """Signed area between the arc and its chord, normalised by chord^2 (same metric as the notebooks)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2:
        return np.nan, np.nan, np.nan
    start, end = np.array([x[0], y[0]]), np.array([x[-1], y[-1]])
    chord_vec = end - start
    chord_length = np.linalg.norm(chord_vec)
    if chord_length <= 1e-12:
        return np.nan, np.nan, np.nan
    points = np.column_stack([x, y])
    relative = points - start
    cross = relative[:, 0] * chord_vec[1] - relative[:, 1] * chord_vec[0]
    distances = np.abs(cross) / chord_length
    seg_lengths = np.linalg.norm(np.diff(points, axis=0), axis=1)
    deviation_area = np.sum(0.5 * (distances[:-1] + distances[1:]) * seg_lengths)
    return deviation_area / (chord_length ** 2), deviation_area, chord_length


# ----------------------------------------------------------------------
# The actual physics: competing-risks integral along an arbitrary path
# ----------------------------------------------------------------------

def compute_xc_curved(
    passer_pos, receiver_pos, defenders, arc_x, arc_y, v0=18.0,
    b0=_DEFAULT_B0, b1=_DEFAULT_B1, factor=_DEFAULT_FACTOR, factor2=_DEFAULT_FACTOR2,
    time_offset_ball=_DEFAULT_TIME_OFFSET_BALL, player_velocity=_DEFAULT_PLAYER_VELOCITY,
    keep_inertial_velocity=_DEFAULT_KEEP_INERTIAL_VELOCITY, use_max=_DEFAULT_USE_MAX,
    v_max=_DEFAULT_V_MAX, a_max=_DEFAULT_A_MAX, inertial_seconds=_DEFAULT_INERTIAL_SECONDS,
    tol_distance=_DEFAULT_TOL_DISTANCE, defender_velocity=None,
    pass_start_location_offset=_DEFAULT_PASS_START_LOCATION_OFFSET,
    use_efficient_sigmoid=_DEFAULT_USE_EFFICIENT_SIGMOID,
):
    """
    Physically-consistent xC / defense-interception-probability for a ball
    travelling along an arbitrary 2D path (arc_x, arc_y) at constant speed v0.

    `defenders` : (N, 2) array of defender (x, y). `defender_velocity`, if given,
    is an (N, 2) array of (vx, vy) - defaults to stationary (0, 0), matching
    what both original notebooks assumed.

    Returns a dict with:
      xc                 : P(receiver possesses the ball) = attack_cum_prob at the final point
      defense_cum_prob   : P(any defender intercepts) at the final point
      cum_p0_final       : P(ball is still "live", uncontested) at the final point
      t_ball, defense_cum_prob_over_path, cum_p0_over_path : full curves, for diagnostics/plots
    """
    arc_x = np.asarray(arc_x, dtype=float)
    arc_y = np.asarray(arc_y, dtype=float)
    defenders = np.asarray(defenders, dtype=float)
    n_def = len(defenders)
    if defender_velocity is None:
        defender_velocity = np.zeros_like(defenders)

    # --- 1. Ball time-to-reach each arc sample: cumulative ARC LENGTH / v0 ---
    # The library's own straight-line model doesn't start timing the ball at the
    # passer's exact position - it starts `pass_start_location_offset` (~-1.52 m,
    # i.e. slightly BEHIND the passer) before t=0, presumably to model backswing/
    # windup. We replicate that by adding this fixed head-start distance to our
    # cumulative arc length before converting to time - this matters a lot here,
    # because the interception sigmoid (b1 ~ -189) is extremely steep, so a
    # ~0.08s timing shift can flip an interception call entirely.
    seg_len = np.hypot(np.diff(arc_x), np.diff(arc_y))
    arc_length = np.concatenate([[0.0], np.cumsum(seg_len)])          # T
    t_ball = (arc_length - pass_start_location_offset) / v0 + time_offset_ball  # T

    # --- 2. Time-to-arrive for the receiver and every defender, at every arc sample ---
    all_pos = np.vstack([receiver_pos, defenders])                     # (1+N) x 2
    all_vel = np.vstack([[0.0, 0.0], defender_velocity])               # receiver treated as stationary
    n_players = all_pos.shape[0]

    tta = approx_two_point_time_to_arrive(
        x=all_pos[:, 0][:, None], y=all_pos[:, 1][:, None],
        vx=all_vel[:, 0][:, None], vy=all_vel[:, 1][:, None],
        x_target=arc_x[None, :], y_target=arc_y[None, :],
        use_max=use_max, velocity=player_velocity, keep_inertial_velocity=keep_inertial_velocity,
        v_max=v_max, a_max=a_max, inertial_seconds=inertial_seconds, tol_distance=tol_distance,
    )  # n_players x T

    # --- 3. Interception RATE at each point (model's own fitted logistic) ---
    diff = tta - t_ball[None, :]                                       # n_players x T
    x = b0 + b1 * diff
    # IMPORTANT: the library defaults to `use_efficient_sigmoid=True`, a fast rational
    # approximation 0.5*(x/(1+|x|)+1) instead of the true logistic 1/(1+exp(-x)). Its tails
    # decay much more slowly (~1/|x| vs ~exp(x)), which is why even defenders far from the
    # pass line contribute small but non-zero interception probability in the real model.
    # Using the *exact* sigmoid here (as one might reasonably assume "the" sigmoid means)
    # silently drops that long-tail contribution and understates defense_cum_prob.
    with np.errstate(over='ignore'):
        sig = _approximate_sigmoid(x) if use_efficient_sigmoid else _sigmoid(x)
    rate = sig / (factor * (v0 ** (-factor2)))                          # n_players x T

    # --- 4. Integrate along the curve (competing-risks / survival model) ---
    def cum_trapz(y, x):
        dx = np.diff(x)
        avg_y = 0.5 * (y[..., :-1] + y[..., 1:])
        integral = np.cumsum(avg_y * dx, axis=-1)
        return np.concatenate([np.zeros(y.shape[:-1] + (1,)), integral], axis=-1)

    sum_rate = rate.sum(axis=0)                                        # T, hazard of ANY player intercepting
    int_sum_rate = cum_trapz(sum_rate[None, :], t_ball)[0]              # T
    cum_p0 = np.exp(-int_sum_rate)                                     # T, P(still uncontested)

    dpr_over_dt = cum_p0[None, :] * rate                                # n_players x T
    pr_cum_prob = cum_trapz(dpr_over_dt, t_ball)                        # n_players x T, P(this player has it by t)

    receiver_cum_prob = pr_cum_prob[0]                                  # T  <-- this IS "xC" (matches library's attack_cum_prob)
    defense_cum_prob_over_path = pr_cum_prob[1:].sum(axis=0)            # T

    return dict(
        xc=float(receiver_cum_prob[-1]),
        defense_cum_prob=float(defense_cum_prob_over_path[-1]),
        cum_p0_final=float(cum_p0[-1]),
        t_ball=t_ball,
        defense_cum_prob_over_path=defense_cum_prob_over_path,
        receiver_cum_prob_over_path=receiver_cum_prob,
        cum_p0_over_path=cum_p0,
    )


# ----------------------------------------------------------------------
# Validation: straight "curve" (bulge=0) must match the library itself
# ----------------------------------------------------------------------

def validate_against_library(passer_pos, receiver_pos, defenders, v0=18.0, verbose=True):
    """Sanity check: a zero-bulge arc must reproduce accessible_space.core.simulate_passes()."""
    passer_pos = np.asarray(passer_pos, dtype=float)
    receiver_pos = np.asarray(receiver_pos, dtype=float)
    defenders = np.asarray(defenders, dtype=float)

    straight_x = np.linspace(passer_pos[0], receiver_pos[0], 400)
    straight_y = np.linspace(passer_pos[1], receiver_pos[1], 400)
    mine = compute_xc_curved(passer_pos, receiver_pos, defenders, straight_x, straight_y, v0=v0)

    # Official library call, single straight phi, fine radial grid for a fair comparison
    all_players = [[passer_pos[0], passer_pos[1], 0., 0.], [receiver_pos[0], receiver_pos[1], 0., 0.]]
    for d in defenders:
        all_players.append([d[0], d[1], 0., 0.])
    PLAYER_POS = np.array([all_players], dtype=np.float32)
    BALL_ARR = np.array([[passer_pos[0], passer_pos[1]]], dtype=np.float32)
    phi_target = float(np.arctan2(receiver_pos[1] - passer_pos[1], receiver_pos[0] - passer_pos[0]))
    phi_grid = np.array([[phi_target]], dtype=np.float32)
    v0_grid = np.array([[v0]], dtype=np.float32)
    teams = [0, 0] + [1] * len(defenders)
    names = ['A', 'B'] + [f'D{i}' for i in range(len(defenders))]

    res = simulate_passes(
        PLAYER_POS, BALL_ARR, phi_grid, v0_grid, np.array([0]), np.array(teams),
        players=np.array(names), passers=np.array(['A']), exclude_passer=True,
        fields_to_return=['attack_cum_prob', 'defense_cum_prob'], radial_gridsize=0.5,
    )
    tdist = float(np.linalg.norm(receiver_pos - passer_pos))
    r_idx = int(np.argmin(np.abs(res.r_grid - tdist)))
    lib_xc = float(res.attack_cum_prob[0, 0, r_idx])
    lib_def = float(res.defense_cum_prob[0, 0, r_idx])

    if verbose:
        print(f"[validation] my xC={mine['xc']:.5f}  vs  library attack_cum_prob={lib_xc:.5f}")
        print(f"[validation] my defense_cum_prob={mine['defense_cum_prob']:.5f}  vs  library={lib_def:.5f}")
    return mine['xc'], lib_xc, mine['defense_cum_prob'], lib_def


## Validation against the library

In [2]:
passer = [-25.0, -29.0]
receiver = [-9.5, -27.5]
defenders = [
    [-33.0, 7.3], [-31.0, 0.0], [-21.0, -12.0], [-31.0, -17.0],
    [-20.69, -28.4], [-33.0, -22.5], [-35.0, -26.0], [-36.0, -28.0], [-14.0, -33.0],
]
validate_against_library(passer, receiver, defenders, v0=18.0)

import numpy as np
np.random.seed(0)
for _ in range(3):
    p = np.random.uniform(-40, 40, 2); r = p + np.random.uniform(-20, 20, 2)
    d = p + np.random.uniform(-15, 15, (6, 2)); v0 = np.random.uniform(12, 25)
    validate_against_library(p, r, d, v0=v0)


[validation] my xC=0.27558  vs  library attack_cum_prob=0.27708
[validation] my defense_cum_prob=0.69059  vs  library=0.68468
[validation] my xC=0.05962  vs  library attack_cum_prob=0.06035
[validation] my defense_cum_prob=0.01967  vs  library=0.02327
[validation] my xC=0.81087  vs  library attack_cum_prob=0.80124
[validation] my defense_cum_prob=0.06164  vs  library=0.06542
[validation] my xC=0.01507  vs  library attack_cum_prob=0.01646
[validation] my defense_cum_prob=0.02940  vs  library=0.03433


## Note on the scenario data

The original `best_pass_scenario.json` (loaded by your two notebooks) wasn't included in the files you uploaded, so the scenario below uses coordinates re-digitized from your plot images — the blocker position (index 4, `[-20.69, -28.4]`) matches your code comment exactly; the other 8 defenders are approximate. **Swap in your real `best_pass_scenario.json` values and everything else below works unchanged.**

Two additional, different events are included too (a fast long diagonal switch, and a slow throughball flanked by two defenders on either side of the lane), as requested — to see the curve-vs-straight trade-off show up in different geometries and speeds.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

BG, PITCH_CLR, LINE_CLR = '#0D1117', '#152315', '#2d5a2d'
C_ABOVE, C_BELOW, C_STRT, C_DEF = '#00D4FF', '#FF6B35', '#FFFFFF', '#FF3333'


def draw_pitch(ax):
    kw = dict(color=LINE_CLR, lw=1.2, alpha=0.55)
    ax.plot([-52.5, 52.5], [-34, -34], **kw); ax.plot([-52.5, 52.5], [34, 34], **kw)
    ax.plot([-52.5, -52.5], [-34, 34], **kw); ax.plot([52.5, 52.5], [-34, 34], **kw)
    ax.plot([0, 0], [-34, 34], **kw); ax.add_patch(plt.Circle((0, 0), 9.15, fill=False, **kw))
    ax.plot([34, 52.5], [16.5, 16.5], **kw); ax.plot([34, 52.5], [-16.5, -16.5], **kw)
    ax.plot([34, 34], [-16.5, 16.5], **kw)


def run_scenario(title, passer, receiver, defenders, v0, bulge_values, blocker_idx=None, fname=None):
    passer, receiver, defenders = map(lambda a: np.array(a, dtype=float), (passer, receiver, defenders))
    pass_labels = [f"Below-{i}" for i in range(len(bulge_values)//2, 0, -1)] + ["Straight"] + \
                  [f"Above+{i}" for i in range(1, len(bulge_values)//2 + 1)]

    passes = []
    for bulge, label in zip(bulge_values, pass_labels):
        ax_, ay_, _ = bezier_arc(passer, receiver, bulge)
        curv, _, _ = calculate_curvature_2d(ax_, ay_)
        out = compute_xc_curved(passer, receiver, defenders, ax_, ay_, v0=v0)
        ptype = 'curved_below' if bulge < 0 else ('curved_above' if bulge > 0 else 'straight')
        passes.append(dict(label=label, bulge=bulge, type=ptype, arc_x=ax_, arc_y=ay_,
                            curvature=curv, xc=out['xc'], def_prob=out['defense_cum_prob']))
        print(f"[{title}] [{ptype:13s}] bulge={bulge:+.2f} | C={curv:.4f} | xC={out['xc']:.4f} | defense={out['defense_cum_prob']:.4f}")

    above_list = [p for p in passes if p['type'] == 'curved_above']
    below_list = [p for p in passes if p['type'] == 'curved_below']
    n_side = max(len(above_list), 1)
    above_alphas = np.linspace(0.30, 1.00, len(above_list)) if above_list else []
    below_alphas = np.linspace(1.00, 0.30, len(below_list)) if below_list else []

    fig = plt.figure(figsize=(20, 11), facecolor=BG)
    ax = fig.add_axes([0.04, 0.15, 0.62, 0.77]); ax.set_facecolor(PITCH_CLR)
    draw_pitch(ax)
    pad = max(8, 0.6 * abs(receiver[0] - passer[0]))
    ax.set_xlim(min(passer[0], receiver[0]) - pad, max(passer[0], receiver[0]) + pad)
    ax.set_ylim(min(passer[1], receiver[1]) - pad, max(passer[1], receiver[1]) + pad)

    for i, d in enumerate(defenders):
        ax.plot(*d, 'o', color=C_DEF, ms=14, zorder=7, mec='white', mew=1.5)
        ax.text(d[0], d[1], 'D', ha='center', va='center', fontsize=8, fontweight='bold', color='#111', zorder=8)
        if blocker_idx is not None and i == blocker_idx:
            ax.add_patch(plt.Circle((d[0], d[1]), 2.0, color=C_DEF, alpha=0.3, zorder=2))
            ax.text(d[0], d[1]-2.5, 'BLOCKER', ha='center', va='top', fontsize=9, color=C_DEF, fontweight='bold',
                    path_effects=[pe.withStroke(linewidth=2.0, foreground=BG)])

    for p in passes:
        if p['type'] == 'curved_above':
            rank = above_list.index(p); color, alpha, lw = C_ABOVE, above_alphas[rank], 1.4 + rank * 0.45
        elif p['type'] == 'curved_below':
            rank = below_list.index(p); color, alpha, lw = C_BELOW, below_alphas[rank], 1.4 + (len(below_list)-1-rank) * 0.45
        else:
            color, alpha, lw = C_STRT, 1.0, 3.0
        ax.plot(p['arc_x'], p['arc_y'], color=color, alpha=alpha, lw=lw, zorder=3)
        mid = len(p['arc_x']) // 2
        dx, dy = p['arc_x'][mid+3]-p['arc_x'][mid-3], p['arc_y'][mid+3]-p['arc_y'][mid-3]
        ax.annotate('', xy=(p['arc_x'][mid]+dx*0.9, p['arc_y'][mid]+dy*0.9),
                    xytext=(p['arc_x'][mid]-dx*0.1, p['arc_y'][mid]-dy*0.1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=min(alpha+0.15, 1.0)), zorder=5)

    ax.plot([passer[0], receiver[0]], [passer[1], receiver[1]], '--', color='#888', lw=1.0, alpha=0.3, zorder=2)
    for pos, ltr, name, color in [(passer, 'A', 'Passer', '#FFDD57'), (receiver, 'B', 'Receiver', '#57FF8A')]:
        ax.plot(*pos, 'o', color=color, ms=16, zorder=8, mec='white', mew=2)
        ax.text(pos[0], pos[1], ltr, ha='center', va='center', fontsize=9, fontweight='bold', color='#111', zorder=9)
        ax.text(pos[0], pos[1]-2.7, name, ha='center', va='top', fontsize=8, color=color, fontweight='bold',
                path_effects=[pe.withStroke(linewidth=2.0, foreground=BG)])
    ax.set_xlabel('x (metres)', color='#AAA'); ax.set_ylabel('y (metres)', color='#AAA')
    ax.tick_params(colors='#888')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.grid(True, color='#1e2d1e', lw=0.5, alpha=0.6, zorder=0)
    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=10)

    ax2 = fig.add_axes([0.70, 0.15, 0.27, 0.77]); ax2.set_facecolor('#141820')
    labels_bar = [p['label'] for p in passes]; xc_vals = [p['xc'] for p in passes]; curv_vals = [p['curvature'] for p in passes]
    bar_colors = []
    for p in passes:
        if p['type'] == 'curved_above': bar_colors.append((0.0, 0.83, 1.0, above_alphas[above_list.index(p)]))
        elif p['type'] == 'curved_below': bar_colors.append((1.0, 0.42, 0.21, below_alphas[below_list.index(p)]))
        else: bar_colors.append((1.0, 1.0, 1.0, 0.9))
    y_pos = np.arange(len(passes))
    ax2.barh(y_pos, xc_vals, color=bar_colors, edgecolor='#333', lw=0.5, height=0.74)
    for i, (xc, curv) in enumerate(zip(xc_vals, curv_vals)):
        ax2.text(min(xc+0.01, max(xc_vals)*1.05), i, f'{xc:.3f}', va='center', ha='left', fontsize=8.5, color='#EEE', fontweight='bold')
        ax2.text(-0.01*max(xc_vals), i, f'C={curv:.3f}', va='center', ha='right', fontsize=7, color='#999')
    ax2.set_yticks(y_pos); ax2.set_yticklabels(labels_bar, fontsize=9, color='#CCC')
    ax2.set_xlim(-0.05*max(xc_vals), max(xc_vals)*1.25)
    ax2.set_xlabel('xC = attack_cum_prob (validated vs. library)', color='#AAA', fontsize=9)
    ax2.set_title('Pass Completion\nProbability (xC)', color='white', fontsize=11, pad=8)
    ax2.tick_params(colors='#888')
    for sp in ax2.spines.values(): sp.set_edgecolor('#333')
    ax2.grid(axis='x', color='#1a2030', lw=0.5, alpha=0.6); ax2.invert_yaxis()

    fig.text(0.5, 0.96, title, ha='center', fontsize=17, fontweight='bold', color='white')
    fig.text(0.5, 0.92, 'Corrected xC: full competing-risks integral along the ACTUAL curved path (validated against accessible-space core)',
              ha='center', fontsize=10.5, color='#AAA')
    plt.savefig(fname, dpi=170, bbox_inches='tight', facecolor=BG, edgecolor='none')
    plt.close(fig)
    return passes


# ---------------------------------------------------------------------------------
# Scenario 1: reconstruction of the original "blocked lane" scenario (approximate
# re-digitized coordinates from your uploaded plot - the original best_pass_scenario.json
# wasn't included in the upload, so swap PASSER_POS/RECEIVER_POS/DEFENDERS below for
# your real data; everything else (the physics) stays exactly the same).
# ---------------------------------------------------------------------------------
passer1 = [-25.0, -29.0]
receiver1 = [-9.5, -27.5]
defenders1 = [
    [-33.0, 7.3], [-31.0, 0.0], [-21.0, -12.0], [-31.0, -17.0],
    [-20.69, -28.4],  # blocker, sitting almost exactly on the direct lane
    [-33.0, -22.5], [-35.0, -26.0], [-36.0, -28.0], [-14.0, -33.0],
]
run_scenario(
    "DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario",
    passer1, receiver1, defenders1, v0=18.0,
    bulge_values=[-0.60, -0.45, -0.30, -0.15, -0.05, 0.00, 0.05, 0.15, 0.30, 0.45, 0.60],
    blocker_idx=4, fname="../figures/das_study/curved_pass_1_original_scenario.png",
)

# ---------------------------------------------------------------------------------
# Scenario 2: "different event" — a long diagonal switch of play with a single
# covering center-back sitting directly on the lane, at higher pass speed
# ---------------------------------------------------------------------------------
passer2 = [-40.0, -20.0]
receiver2 = [10.0, 15.0]
defenders2 = [
    [-15.0, -2.0],   # sits almost exactly on the straight lane
    [-25.0, 10.0], [5.0, -5.0], [-5.0, 20.0], [15.0, 5.0], [-30.0, -15.0],
]
run_scenario(
    "DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)",
    passer2, receiver2, defenders2, v0=24.0,
    bulge_values=[-0.5, -0.35, -0.2, -0.08, -0.02, 0.00, 0.02, 0.08, 0.2, 0.35, 0.5],
    blocker_idx=0, fname="../figures/das_study/curved_pass_2_long_switch.png",
)

# ---------------------------------------------------------------------------------
# Scenario 3: "different event" — a tight, slow throughball into the box with two
# defenders flanking the lane (tests whether curving TOWARD one defender while
# avoiding the other still produces the expected trade-off)
# ---------------------------------------------------------------------------------
passer3 = [30.0, 0.0]
receiver3 = [48.0, 3.0]
defenders3 = [
    [39.0, 4.0],    # just above the lane
    [39.0, -3.0],   # just below the lane
    [44.0, 8.0], [44.0, -8.0], [35.0, 10.0],
]
run_scenario(
    "DAS Curved Pass Study (Corrected) — Slow Throughball Between Two Flanking Defenders (v0=14 m/s)",
    passer3, receiver3, defenders3, v0=14.0,
    bulge_values=[-0.35, -0.25, -0.15, -0.07, -0.02, 0.00, 0.02, 0.07, 0.15, 0.25, 0.35],
    blocker_idx=None, fname="../figures/das_study/curved_pass_3_flanked_throughball.png",
)

print("\nAll three scenarios written to ../figures/das_study/")


[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.60 | C=0.2255 | xC=0.1834 | defense=0.8095
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.45 | C=0.1613 | xC=0.1995 | defense=0.7884
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.30 | C=0.1035 | xC=0.2217 | defense=0.7589
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.15 | C=0.0504 | xC=0.2493 | defense=0.7226
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.05 | C=0.0167 | xC=0.2675 | defense=0.7001
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [straight     ] bulge=+0.00 | C=0.0000 | xC=0.2755 | defense=0.6906
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_above ] bulge=+0.05 | C=0.0167 | xC=0.2827 | defense=0.6829
[DAS Curved Pass Study (Corrected) — Orig

[DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)] [curved_below ] bulge=-0.50 | C=0.1819 | xC=0.0059 | defense=0.9940
[DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)] [curved_below ] bulge=-0.35 | C=0.1221 | xC=0.0053 | defense=0.9945
[DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)] [curved_below ] bulge=-0.20 | C=0.0677 | xC=0.0022 | defense=0.9977
[DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)] [curved_below ] bulge=-0.08 | C=0.0267 | xC=0.0032 | defense=0.9966
[DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)] [curved_below ] bulge=-0.02 | C=0.0067 | xC=0.0034 | defense=0.9964
[DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)] [straight     ] bulge=+0.00 | C=0.0000 | xC=0.0035 | defense=0.9963
[DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)] [cur

[DAS Curved Pass Study (Corrected) — Slow Throughball Between Two Flanking Defenders (v0=14 m/s)] [curved_below ] bulge=-0.35 | C=0.1221 | xC=0.0665 | defense=0.9334
[DAS Curved Pass Study (Corrected) — Slow Throughball Between Two Flanking Defenders (v0=14 m/s)] [curved_below ] bulge=-0.25 | C=0.0854 | xC=0.0794 | defense=0.9205
[DAS Curved Pass Study (Corrected) — Slow Throughball Between Two Flanking Defenders (v0=14 m/s)] [curved_below ] bulge=-0.15 | C=0.0504 | xC=0.0824 | defense=0.9175
[DAS Curved Pass Study (Corrected) — Slow Throughball Between Two Flanking Defenders (v0=14 m/s)] [curved_below ] bulge=-0.07 | C=0.0234 | xC=0.0890 | defense=0.9108
[DAS Curved Pass Study (Corrected) — Slow Throughball Between Two Flanking Defenders (v0=14 m/s)] [curved_below ] bulge=-0.02 | C=0.0067 | xC=0.0847 | defense=0.9150
[DAS Curved Pass Study (Corrected) — Slow Throughball Between Two Flanking Defenders (v0=14 m/s)] [straight     ] bulge=+0.00 | C=0.0000 | xC=0.0821 | defense=0.9177
[DAS


All three scenarios written to /mnt/user-data/outputs/
